# 🩺 Multimodal CDSS: ResNet-50 Deep Learning Training Notebook
### VTU 5th Sem Mini Project — Dept. of CSE (Data Science)
**Objective:** Fine-tune ResNet-50 on Chest Radiographs (X-rays) for Pulmonary Opacity & Pneumonia detection using Transfer Learning and evaluate clinical metrics (Loss, Accuracy, Confusion Matrix, Sensitivity/Recall, Specificity).

In [ ]:
# Cell 1: Hardware Acceleration Verification (GPU Check)
import torch

print("PyTorch Version:", torch.__version__)
if torch.cuda.is_available():
    print(f"🚀 GPU Detected: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda:0")
else:
    print("⚠️ GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU")
    device = torch.device("cpu")

In [ ]:
# Cell 2: Download Chest X-Ray Dataset
import os

# Remove any previously corrupted or broken download
if os.path.exists('chest_xray.zip'):
    if os.path.getsize('chest_xray.zip') < 1000000:
        os.remove('chest_xray.zip')

print("[*] Downloading Chest X-Ray dataset (~1.1 GB)...")
# Direct verified high-speed mirror of Kaggle Chest X-Ray Pneumonia dataset
!wget -c --show-progress https://www.piclab.ai/classes/cv2020/chest_xray.zip -O chest_xray.zip

print("[*] Extracting files...")
!unzip -q -o chest_xray.zip -d ./dataset/

# Dynamically locate dataset root folder
base_path = './dataset/chest_xray'
for candidate in ['./dataset/chest_xray', './dataset', './dataset/chest-xray', './chest_xray']:
    if os.path.exists(os.path.join(candidate, 'train')):
        base_path = candidate
        break

print(f"✓ Using dataset directory: {base_path}")
for split in ['train', 'val', 'test']:
    p = os.path.join(base_path, split)
    if os.path.exists(p):
        classes = [c for c in os.listdir(p) if os.path.isdir(os.path.join(p, c))]
        print(f"✓ Found {split} split with classes: {classes}")

In [ ]:
# Cell 3: Data Augmentation & PyTorch DataLoaders
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Clinical Data Transforms: ImageNet normalization & subtle geometric augmentation
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=8),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
}

batch_size = 32
image_datasets = {}
dataloaders = {}
dataset_sizes = {}

for phase in ['train', 'val', 'test']:
    phase_dir = os.path.join(base_path, phase)
    if os.path.exists(phase_dir):
        image_datasets[phase] = datasets.ImageFolder(phase_dir, data_transforms[phase])
        dataloaders[phase] = DataLoader(
            image_datasets[phase], 
            batch_size=batch_size, 
            shuffle=(phase == 'train'),
            num_workers=2,
            pin_memory=True
        )
        dataset_sizes[phase] = len(image_datasets[phase])
        print(f"Phase '{phase}': {dataset_sizes[phase]} images")

class_names = image_datasets['train'].classes
print(f"\nTarget Clinical Classes: {class_names}")

In [ ]:
# Cell 4: Instantiate ResNet-50 with Custom Clinical MLP Head
import torch.nn as nn
from torchvision import models

print("[*] Loading pre-trained ResNet-50 backbone...")
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze the convolutional feature extraction backbone (Transfer Learning)
for param in model.parameters():
    param.requires_grad = False

# Custom Clinical Classification Head matching the CDSS application
in_features = model.fc.in_features  # 2048
model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, 128),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(128, len(class_names))
)

model = model.to(device)
print(f"✓ Custom ResNet-50 ready on {device} (Classes: {len(class_names)})")

In [ ]:
# Cell 5: Train the Model (Cross-Entropy Loss & Adam Optimizer)
import torch.optim as optim
import time
import copy

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

num_epochs = 5
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

eval_phase = 'val' if 'val' in dataloaders else 'test'
start_time = time.time()

for epoch in range(num_epochs):
    print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")
    for phase in ['train', eval_phase]:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        if phase == 'train':
            scheduler.step()

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = (running_corrects.double() / dataset_sizes[phase]).item()

        history[f"{phase}_loss" if phase == 'train' else 'val_loss'].append(epoch_loss)
        history[f"{phase}_acc" if phase == 'train' else 'val_acc'].append(epoch_acc)
        print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc*100:.2f}%")

        if phase != 'train' and epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())

elapsed = (time.time() - start_time) / 60
print(f"\n🎉 Training completed in {elapsed:.2f} minutes! Best Validation Accuracy: {best_acc*100:.2f}%")
model.load_state_dict(best_model_wts)

In [ ]:
# Cell 6: Detailed Test Evaluation, Confusion Matrix & Metrics
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

test_phase = 'test' if 'test' in dataloaders else eval_phase
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for inputs, labels in dataloaders[test_phase]:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Compute Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
print("=== CLASSIFICATION REPORT ===")
print(classification_report(all_labels, all_preds, target_names=class_names))

# Plot Publication-Quality Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Clinical Triage Confusion Matrix (ResNet-50)', fontsize=12, fontweight='bold')
plt.ylabel('Actual Clinical Ground Truth')
plt.xlabel('AI Predicted Category')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
# Cell 7: Plot Training & Validation Curves (For Report Chapter 5)
plt.figure(figsize=(12, 4))

# Loss Plot
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], 'o-', label='Training Loss', color='#0284c7')
plt.plot(history['val_loss'], 's-', label='Validation Loss', color='#dc2626')
plt.title('Cross-Entropy Loss vs. Epochs', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

# Accuracy Plot
plt.subplot(1, 2, 2)
plt.plot([a * 100 for a in history['train_acc']], 'o-', label='Training Accuracy', color='#16a34a')
plt.plot([a * 100 for a in history['val_acc']], 's-', label='Validation Accuracy', color='#9333ea')
plt.title('Classification Accuracy vs. Epochs (%)', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300)
plt.show()

In [ ]:
# Cell 8: Save Model Weights & Download Everything to Your Laptop
from google.colab import files

# Save trained PyTorch state dict compatible with app.py
output_weights_name = "cdss_resnet50.pth"
torch.save(model.state_dict(), output_weights_name)
print(f"✅ Successfully saved weights to {output_weights_name}!")

# Trigger auto-downloads in browser
files.download(output_weights_name)
files.download('confusion_matrix.png')
files.download('training_curves.png')
print("📥 Download triggered for: weights (.pth) and evaluation plots (.png)!")